# Options Pricing Walkthrough

This notebook is a short companion to the web app. It explains the Black-Scholes assumptions, validates a standard reference case, and demonstrates how the backend pricing engine can be reused directly from Python.

## What this notebook covers

- European option pricing with the Black-Scholes model
- Greeks for first-order risk interpretation
- A reference-value sanity check
- A small Monte Carlo scenario using the same backend code


## Model assumptions

The MVP uses the Black-Scholes framework for European options. It assumes:

- lognormal price dynamics
- constant volatility and interest rates
- frictionless markets with continuous trading
- no early exercise

These assumptions make the model useful for baseline valuation and sensitivity analysis, but not a market predictor.


In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve().parent
backend_root = project_root / "backend"

if str(backend_root) not in sys.path:
    sys.path.append(str(backend_root))

from app.schemas import MonteCarloInput, OptionType, PricingInput
from app.services.pricing import black_scholes_price, compute_greeks, run_monte_carlo

inputs = PricingInput(
    spot_price=100,
    strike_price=100,
    time_to_expiry=1,
    volatility=0.2,
    risk_free_rate=0.05,
    option_type=OptionType.CALL,
)

price = black_scholes_price(inputs)
greeks = compute_greeks(inputs)

print(f"Reference call price: {price.option_price:.4f}")
print(f"Delta: {greeks.delta:.4f}")
print(f"Gamma: {greeks.gamma:.4f}")
print(f"Theta: {greeks.theta:.4f}")
print(f"Vega: {greeks.vega:.4f}")


The standard Black-Scholes reference case above should produce a call price close to `10.4506`. Matching that benchmark is a quick sanity check that the pricing engine is implemented correctly.


In [ ]:
simulation_inputs = MonteCarloInput(
    **inputs.model_dump(),
    num_simulations=3000,
    num_steps=30,
    random_seed=42,
)

simulation = run_monte_carlo(simulation_inputs)
simulation
